In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from pathlib import Path
import re
import unicodedata # for ludmin's dataset

from sklearn.metrics import mean_squared_error, mean_absolute_error

# sentiment library

from rapidfuzz import fuzz, process

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox

os.getcwd()

'/Users/mercyeme/Documents/Data_Projects_2026/RecessionPop'

In [2]:
# loading BBHot100 dataset
billboard = pd.read_csv("data/raw/hot100.csv")

In [3]:
# getting a sense of the BBHot100 data
print("Shape:", billboard.shape)
print("Columns:", billboard.columns.tolist())

# parse date
billboard["Date"] = pd.to_datetime(billboard["Date"])

display(billboard.head())

print(f"First chart date: {billboard['Date'].min()}") # learning different text formats and checking date range
print("Last chart date:", billboard["Date"].max())


Shape: (354858, 8)
Columns: ['Date', 'Song', 'Artist', 'Rank', 'Last Week', 'Peak Position', 'Weeks in Charts', 'Image URL']


,Date,Song,Artist,Rank,Last Week,Peak Position,Weeks in Charts,Image URL
0,1958-07-30,Poor Little Fool,Ricky Nelson,1,-,1,1,NaN
1,1958-07-30,Patricia,Perez Prado And His Orchestra,2,-,2,1,NaN
2,1958-07-30,Splish Splash,Bobby Darin,3,-,3,1,NaN
3,1958-07-30,Hard Headed Woman,Elvis Presley With The Jordanaires,4,-,4,1,https://charts-static.billboard.com/img/1958/0...
4,1958-07-30,When,Kalin Twins,5,-,5,1,NaN


First chart date: 1958-07-30 00:00:00
Last chart date: 2026-07-29 00:00:00


In [4]:
# extracting song and artist name from BBHot100 dataset DF only for new DF
billboard_songs = (
    billboard[
        ["Song", "Artist"]
        ]
    .drop_duplicates() # to get rid of songs with multiple BBHot100 entries (ex. All I Want for Christmas by MC). Only need one instance of the song.
    .reset_index(drop=True))

len(billboard_songs)

print(f"Weekly Billboard rows:, {len(billboard)}")
print("Unique song-artist pairs:", len(billboard_songs))

display(billboard_songs.head()) # table to be used to match with 550K dataset

Weekly Billboard rows:, 354858
Unique song-artist pairs: 32592


,Song,Artist
0,Poor Little Fool,Ricky Nelson
1,Patricia,Perez Prado And His Orchestra
2,Splish Splash,Bobby Darin
3,Hard Headed Woman,Elvis Presley With The Jordanaires
4,When,Kalin Twins


In [5]:
# loading 550k Audio Features analysis dataset as just a file path (not DF)
songs_path = "data/raw/songs.csv"

In [6]:
# checking column names
song_preview = pd.read_csv(
    songs_path,
    nrows=5
)

display("Columns:", song_preview.columns.tolist()) # for easier readability
display(song_preview)

'Columns:'

['id',
 'name',
 'album_name',
 'artists',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'duration_ms',
 'lyrics',
 'year',
 'genre',
 'popularity',
 'total_artist_followers',
 'avg_artist_popularity',
 'artist_ids',
 'niche_genres']

,id,name,album_name,artists,danceability,energy,key,loudness,mode,speechiness,...,tempo,duration_ms,lyrics,year,genre,popularity,total_artist_followers,avg_artist_popularity,artist_ids,niche_genres
0,0Prct5TDjAnEgIqbxcldY9,!,UNDEN!ABLE,"[""HELLYEAH""]",0.415,0.605,7,-11.157,1,0.0575,...,100.059,79500,"He said he came from Jamaica,\nhe owned a coup...",2016,Rock,0,769490,52.0,"[""4hxDvVq5t8ebPYPdBl1F9f""]","[""groove metal"", ""metal""]"
1,2ASl4wirkeYm3OWZxXKYuq,!!,Childhood Dreams,"[""Yxngxr1""]",0.788,0.648,7,-9.135,0,0.3150,...,79.998,114000,"Fuck the bitch, now she running with my kids\n...",2019,Hip-Hop,29,143628,45.0,"[""2jwRHcdgkRhelYEMqndDKe""]",[]
2,5tA3ImW310llKo8EMBj2Ga,!!Noble Stabbings!!,Situationist Comedy,"[""Dillinger Four""]",0.171,0.957,2,-5.749,1,0.1490,...,175.317,197400,You like to stand on the other side\nPoint and...,2002,Rock,0,36619,35.0,"[""4YAN46l70QV0PGXlMg0iHi""]","[""melodic hardcore"", ""pop punk"", ""punk"", ""skat..."
3,0fROT4kK5oTm8xO8PX6EJF,!I'll Be Back!,!I'll Be Back!,"[""Ril\u00e8s""]",0.823,0.612,1,-7.767,1,0.2480,...,142.959,178533,"It's been a while, shit, I missed the rehab, p...",2018,Hip-Hop,43,929303,63.0,"[""6pdcQa7by8IKuoVXvgknlI""]","[""french rap""]"
4,1xBFhv5faebv3mmwxx7DnS,!Lost!,!Lost!,"[""Ril\u00e8s""]",0.729,0.552,7,-8.562,0,0.0650,...,86.103,186197,I would like to give you all my time\nI would ...,2018,Hip-Hop,0,929303,63.0,"[""6pdcQa7by8IKuoVXvgknlI""]","[""french rap""]"


In [7]:
# exclude lyrics column for now
song_cols = [
    'id',
 'name',
 'album_name',
 'artists',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'duration_ms',
 'year',
 'genre',
 'popularity',
 'total_artist_followers',
 'avg_artist_popularity',
 'artist_ids',
 'niche_genres'
]

In [8]:
# load chunks to preserve memory
chunks = []

# Read songs.csv 50,000 rows at a time, and only load the columns listed in song_cols. Each chunk = DF
for chunk in pd.read_csv(
    songs_path,
    usecols=song_cols,
    chunksize=50_000
):
    chunks.append(chunk)

# takes all those separate DFs and stacks them vertically into one big DF called songs.
songs = pd.concat(
    chunks,
    ignore_index=True # for clean "new" default indexes for new dataframe
)

In [9]:
# save feature engineered songs DF to processed folder

songs.to_parquet(
    "data/processed/song_metadata.parquet",
    index=False
)

# to load in the future: 
# songs = pd.read_parquet(
#    "data/processed/song_metadata.parquet"
#)

In [10]:
# 550k dataset: number of songs charted by genre

songs["genre"].value_counts(
    dropna=False
)

genre
Rock          197168
Pop            72539
Electronic     69755
Folk           50843
Country        46076
Hip-Hop        41745
R&B            27337
Jazz           17010
Blues          15976
Classical      12173
Name: count, dtype: int64

In [11]:
# 550k dataset: number of songs charted by year

songs["year"].value_counts().sort_index()

year
1900       30
1917        3
1923       29
1924        5
1925       15
        ...  
2021    14797
2022    19705
2023     6182
2024        1
2025        2
Name: count, Length: 105, dtype: int64

In [12]:
print("Earliest year:", songs["year"].min())
print("Latest year:", songs["year"].max())
# what??? 1900???

Earliest year: 1900
Latest year: 2025


In [13]:
# audit found inconsistencies in genre and year classification pre 1917. Look at how dancehall is classified LMAOOOO

display(
    songs.loc[
        songs["year"] < 1920,
        ["name", "artists", "year", "genre", "niche_genres"]
    ].sort_values("year")
    .sample(
        min(10, (songs["year"] < 1920).sum())
    )
)

,name,artists,year,genre,niche_genres
170108,Gimme the Light (feat. Busta Rhymes) - Pass th...,"[""Sean Paul"", ""Busta Rhymes"", ""Jigzagula"", ""Mu...",1900,Folk,"[""dancehall"", ""east coast hip hop""]"
226117,I'm Still in Love with You (feat. Sasha),"[""Sean Paul"", ""Sasha"", ""Jeremy Harding"", ""Murr...",1900,Folk,"[""dancehall"", ""jungle"", ""ragga"", ""reggae"", ""ri..."
171171,Give It Up to Me,"[""Sean Paul""]",1900,Folk,"[""dancehall""]"
408594,Smoke Weed Everyday,"[""Anthony B""]",1900,Folk,"[""dancehall"", ""ragga"", ""reggae"", ""riddim"", ""ro..."
426342,Stopper,"[""Cutty Ranks""]",1900,Folk,"[""dancehall"", ""jungle"", ""lovers rock"", ""ragga""..."
405848,Sledgehammer,"[""Peter Gabriel""]",1900,Rock,"[""art rock""]"
296058,Mercy Street,"[""Peter Gabriel""]",1900,Rock,"[""art rock""]"
167118,Get Busy,"[""Sean Paul""]",1900,Folk,"[""dancehall""]"
178729,Got 2 Luv U (feat. Alexis Jordan),"[""Sean Paul"", ""Alexis Jordan""]",1900,Folk,"[""dancehall""]"
177306,Good Old Dancehall Vibes (feat. Big Youth),"[""Beres Hammond"", ""Big Youth""]",1900,Folk,"[""dancehall"", ""dub"", ""lovers rock"", ""ragga"", ""..."


In [14]:
# all other years post 1917 seem fine but need to feature eng genres using niche genres column
display(
    songs.loc[
        songs["year"] <= 1999,
        ["name", "artists", "year", "genre", "niche_genres"]
    ].sample(
        min(10, (songs["year"] <= 1999).sum())
    )
)

,name,artists,year,genre,niche_genres
454530,The Fishin' Hole,"[""Andy Griffith""]",1961,Country,"[""christian bluegrass"", ""country christian"", ""..."
331577,"Oh, Sister","[""Jimmy LaFave""]",1999,Country,"[""alt country"", ""americana"", ""red dirt""]"
412389,Somebody In Your Life,"[""Peabo Bryson""]",1999,R&B,"[""quiet storm""]"
351769,Pity For A Dime,"[""Creed""]",1997,Rock,"[""post-grunge""]"
468770,The Spin,"[""Scientists""]",1983,Rock,"[""garage rock"", ""noise rock"", ""post-punk"", ""po..."
437837,Tahitian Moon,"[""Porno for Pyros""]",1996,R&B,"[""funk rock"", ""grunge""]"
377298,Rockin' After Midnight,"[""Marvin Gaye""]",1982,R&B,"[""classic soul"", ""motown"", ""northern soul"", ""q..."
266101,Let's Face The Music And Dance,"[""Fred Astaire""]",1932,Jazz,"[""adult standards"", ""big band"", ""swing music""]"
6976,"6'1"" (Remastered)","[""Liz Phair""]",1993,Pop,[]
435619,Sweet Love On My Mind,"[""Johnny Burnette & The Rock 'N' Roll Trio""]",1956,Rock,"[""psychobilly"", ""rock and roll"", ""rockabilly""]"


In [15]:
# auditing folk genre (many mislabeled songs found here)
display(
    songs.loc[
        songs["genre"] == "Folk",
        ["name", "artists", "genre", "niche_genres"]
    ]
)

,name,artists,genre,niche_genres
31,"""Blind""","[""SOAK""]",Folk,[]
33,"""Bluejays and Cardinals""","[""The Mountain Goats""]",Folk,"[""anti-folk"", ""folk punk"", ""indie folk""]"
37,"""C"" IS FOR COOKIE","[""Little Apple Band""]",Folk,[]
38,"""C"" Is For Cookie","[""Little Apple Band""]",Folk,[]
39,"""C"" Is for Cookie","[""Little Apple Band""]",Folk,[]
...,...,...,...,...
550561,3ZN,"[""Rygin King""]",Folk,"[""dancehall""]"
550578,4 Estaciones - En Vivo,"[""Guaco""]",Folk,"[""merengue"", ""rumba congolaise""]"
550589,4 Weeks,"[""Left Alone""]",Folk,"[""punk"", ""ska"", ""ska punk"", ""skate punk""]"
550600,40 Hours Ago,"[""Jaymay""]",Folk,"[""anti-folk""]"


In [16]:
# looking at niche genres and the dataset's rationale for classifing each one for the official genres
genre_combo_counts = (
    songs
    .groupby(
        ["genre", "niche_genres"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False
    )
)

display(genre_combo_counts.head(50))

,genre,niche_genres,count
1279,Classical,"[""musicals""]",4929
2443,Country,"[""country""]",4876
17885,Pop,"[""christmas""]",3330
20549,R&B,"[""r&b""]",3176
2289,Country,"[""classic country"", ""country""]",2643
11765,Folk,"[""indie folk""]",2582
18861,Pop,"[""pop punk""]",2543
11999,Folk,"[""singer-songwriter""]",2385
22666,Rock,"[""death metal"", ""grindcore""]",2325
1399,Country,"[""acoustic country"", ""country""]",2203


In [17]:
genre_combo_counts.to_csv(
    "data/processed/genre_combo_counts.csv",
    index=False
)

In [18]:
# Feature eng of genres

base_genre_map = {
    "Pop": "Pop",
    "Rock": "Rock",
    "Hip-Hop": "Hip-Hop",
    "R&B": "R&B/Soul",
    "Country": "Country",
    "Electronic": "Electronic/Dance",
    "Folk": "Folk",
    "Jazz": "Jazz",
    "Classical": "Classical/Musical",
    "Blues": "Blues"
}

songs["genre_original"] = songs["genre"] # preserve OG

songs["genre_engineered"] = (
    songs["genre_original"]
    .map(base_genre_map)
    .fillna("Other")
) # copy for feature eng version



In [19]:
import ast

def parse_genre_list(x):
    if isinstance(x, list):
        return x

    if pd.isna(x):
        return []

    try:
        return ast.literal_eval(x)

    except (ValueError, SyntaxError, TypeError):
        return []


songs["niche_genre_list"] = (
    songs["niche_genres"]
    .apply(parse_genre_list)
)

In [20]:
genre_overrides = {

    "Reggae/Dancehall": {
        "calypso",
        "christian reggae",
        "dancehall",
        "dub",
        "lovers rock",
        "ragga",
        "reggae",
        "reggae pop",
        "reggae rock",
        "rocksteady",
        "roots reggae",
        "ska",
        "soca"
    },

    "Afrobeats/African Pop": {
        "afro adura",
        "afro r&b",
        "afro soul",
        "afrobeat",
        "afrobeats",
        "afrogospel",
        "afropiano",
        "afropop",
        "afroswing",
        "alté",
        "amapiano",
        "asakaa",
        "azonto",
        "bongo flava",
        "ghanaian hip hop",
        "highlife",
        "hiplife",
        "nigerian drill",
        "rumba congolaise"
    },

    "Latin/Tropical": {
        "bachata",
        "bolero",
        "bossa nova",
        "cumbia",
        "forró",
        "latin",
        "latin hip hop",
        "latin jazz",
        "latin pop",
        "mambo",
        "mariachi",
        "merengue",
        "mpb",
        "música mexicana",
        "norteño",
        "pagode",
        "ranchera",
        "reggaeton",
        "salsa",
        "samba",
        "sertanejo",
        "tango",
        "trap latino",
        "urbano latino"
    },

    "R&B/Soul": {
        "alternative r&b",
        "classic soul",
        "dark r&b",
        "indie r&b",
        "indie soul",
        "motown",
        "neo soul",
        "new jack swing",
        "northern soul",
        "philly soul",
        "quiet storm",
        "r&b",
        "retro soul",
        "soul",
        "trap soul"
    },

    "Hip-Hop": {
        "alternative hip hop",
        "boom bap",
        "cloud rap",
        "crunk",
        "drill",
        "east coast hip hop",
        "emo rap",
        "experimental hip hop",
        "g-funk",
        "gangster rap",
        "hardcore hip hop",
        "hip hop",
        "melodic rap",
        "old school hip hop",
        "rap",
        "southern hip hop",
        "trap",
        "underground hip hop",
        "west coast hip hop"
    },

    "Electronic/Dance": {
        "deep house",
        "drum and bass",
        "dubstep",
        "edm",
        "electro house",
        "electronic",
        "eurodance",
        "future bass",
        "hardstyle",
        "house",
        "progressive house",
        "tech house",
        "techno",
        "trance",
        "tropical house"
    },

    "Country": {
        "alt country",
        "americana",
        "bluegrass",
        "classic country",
        "country",
        "country rock",
        "honky tonk",
        "outlaw country",
        "red dirt",
        "texas country",
        "traditional country"
    },

    "Jazz": {
        "bebop",
        "big band",
        "cool jazz",
        "jazz",
        "jazz fusion",
        "smooth jazz",
        "soul jazz",
        "swing music",
        "vocal jazz"
    },

    "Blues": {
        "blues",
        "blues rock",
        "classic blues",
        "country blues",
        "delta blues",
        "modern blues"
    },

    "Classical/Musical": {
        "baroque",
        "chamber music",
        "choral",
        "classical",
        "classical crossover",
        "contemporary classical",
        "musicals",
        "opera",
        "orchestral",
        "renaissance"
    },

    "Folk": {
        "anti-folk",
        "celtic folk",
        "folk",
        "folk pop",
        "folk rock",
        "indie folk",
        "singer-songwriter",
        "traditional folk"
    },

    "Rock": {
        "alternative rock",
        "art rock",
        "classic rock",
        "garage rock",
        "glam rock",
        "grunge",
        "hard rock",
        "indie rock",
        "metal",
        "metalcore",
        "post-grunge",
        "progressive rock",
        "psychedelic rock",
        "punk",
        "rock",
        "rock and roll"
    },

    "Pop": {
        "art pop",
        "bedroom pop",
        "dance pop",
        "electropop",
        "indie pop",
        "j-pop",
        "k-pop",
        "pop",
        "power pop",
        "soft pop",
        "synthpop"
    },
    
    "Indie": {
        "indie",
        "chinese indie",
        "german indie",
        "hindi indie",
        "indian indie",
        "indie québécois",
        "indonesian indie",
        "japanese indie",
        "lo-fi indie",
        "mexican indie",
        "pinoy indie",
        "taiwanese indie",
        "tamil indie",
        "vietnam indie"
}

}

In [21]:
all_niche_tags = set(
    songs["niche_genre_list"]
    .explode()
    .dropna()
    .str.lower()
    .str.strip()
)

override_tags = set().union(
    *genre_overrides.values()
)

covered_tags = all_niche_tags & override_tags
uncovered_tags = all_niche_tags - override_tags

print("Total niche genres:", len(all_niche_tags))
print("Covered by overrides:", len(covered_tags))
print("Not explicitly covered:", len(uncovered_tags))

Total niche genres: 754
Covered by overrides: 188
Not explicitly covered: 566


In [22]:
uncovered_counts = (
    songs["niche_genre_list"]
    .explode()
    .dropna()
    .str.lower()
    .str.strip()
    .value_counts()
)

uncovered_counts = uncovered_counts[
    uncovered_counts.index.isin(uncovered_tags)
]

display(uncovered_counts.head(100))

niche_genre_list
death metal         16148
post-hardcore       14265
pop punk            13001
skate punk          11691
industrial          10991
                    ...  
funk rock            2960
children's music     2914
melodic bass         2908
comedy               2852
psychobilly          2782
Name: count, Length: 100, dtype: int64

In [23]:
tag_genre_counts = (
    songs[["genre_original", "niche_genre_list"]]
    .explode("niche_genre_list")
    .dropna(subset=["niche_genre_list"])
    .assign(
        niche_genre=lambda x:
        x["niche_genre_list"]
        .str.lower()
        .str.strip()
    )
    .groupby(
        ["niche_genre", "genre_original"]
    )
    .size()
    .reset_index(name="count")
)

In [24]:
tag_totals = (
    tag_genre_counts
    .groupby("niche_genre")["count"]
    .sum()
    .rename("total")
)

In [25]:
dominant_genres = (
    tag_genre_counts
    .sort_values("count", ascending=False)
    .drop_duplicates("niche_genre")
    .merge(
        tag_totals,
        on="niche_genre"
    )
)

dominant_genres["dominant_share"] = (
    dominant_genres["count"]
    / dominant_genres["total"]
)

In [26]:
uncovered_audit = (
    dominant_genres[
        dominant_genres["niche_genre"]
        .isin(uncovered_tags)
    ]
    .sort_values(
        "total",
        ascending=False
    )
)

display(uncovered_audit.head(100))

,niche_genre,genre_original,count,total,dominant_share
3,death metal,Rock,16140,16148,0.999505
6,post-hardcore,Rock,14229,14265,0.997476
41,pop punk,Rock,7777,13001,0.598185
14,skate punk,Rock,10095,11691,0.863485
42,industrial,Rock,7753,10991,0.705395
...,...,...,...,...,...
248,funk rock,R&B,1346,2960,0.454730
165,children's music,Folk,2825,2914,0.969458
162,melodic bass,Electronic,2897,2908,0.996217
171,comedy,Pop,2651,2852,0.929523


In [27]:
needs_review = uncovered_audit[
    (uncovered_audit["total"] >= 500)
    & (uncovered_audit["dominant_share"] < 0.80)
]

display(needs_review)

,niche_genre,genre_original,count,total,dominant_share
41,pop punk,Rock,7777,13001,0.598185
42,industrial,Rock,7753,10991,0.705395
111,christian,Pop,4247,10761,0.394666
82,ska punk,Folk,5386,10346,0.520588
75,darkwave,Rock,5544,9941,0.557690
...,...,...,...,...,...
386,miami bass,Electronic,374,594,0.629630
376,soundtrack,Classical,393,590,0.666102
385,cajun,Blues,380,583,0.651801
409,dark ambient,Electronic,273,556,0.491007


In [28]:
exploded_genres = (
    songs[
        [
            "name",
            "artists",
            "genre_original",
            "niche_genre_list"
        ]
    ]
    .explode("niche_genre_list")
    .dropna(subset=["niche_genre_list"])
    .rename(columns={"niche_genre_list": "niche_genre"})
)

exploded_genres["niche_genre"] = (
    exploded_genres["niche_genre"]
    .str.lower()
    .str.strip()
)

In [29]:
genre_distribution = (
    exploded_genres
    .groupby(
        ["niche_genre", "genre_original"]
    )
    .size()
    .unstack(fill_value=0)
)

In [30]:
genre_shares = (
    genre_distribution
    .div(
        genre_distribution.sum(axis=1),
        axis=0
    )
)

In [31]:
genre_distribution.columns = [
    f"{col}_count"
    for col in genre_distribution.columns
]

genre_shares.columns = [
    f"{col}_share"
    for col in genre_shares.columns
]

In [32]:
genre_audit = pd.concat(
    [
        genre_distribution,
        genre_shares
    ],
    axis=1
)

genre_audit["total_count"] = (
    genre_distribution.sum(axis=1)
)

In [33]:
official_counts = (
    exploded_genres
    .groupby(
        ["niche_genre", "genre_original"]
    )
    .size()
    .reset_index(name="count")
)

official_counts["rank"] = (
    official_counts
    .groupby("niche_genre")["count"]
    .rank(
        method="first",
        ascending=False
    )
)

In [34]:
top1 = (
    official_counts[
        official_counts["rank"] == 1
    ]
    .set_index("niche_genre")
)

genre_audit["dominant_genre"] = (
    top1["genre_original"]
)

genre_audit["dominant_share"] = (
    top1["count"]
    / genre_audit["total_count"]
)

In [35]:
top2 = (
    official_counts[
        official_counts["rank"] == 2
    ]
    .set_index("niche_genre")
)

genre_audit["second_genre"] = (
    top2["genre_original"]
)

genre_audit["second_share"] = (
    top2["count"]
    / genre_audit["total_count"]
)

In [36]:
def get_examples(group, n=3):
    examples = (
        group[
            ["name", "artists"]
        ]
        .drop_duplicates()
        .head(n)
    )

    return " | ".join(
        examples["name"].astype(str)
        + " — "
        + examples["artists"].astype(str)
    )

In [37]:
examples = (
    exploded_genres
    .groupby("niche_genre")[["name", "artists"]]
    .apply(get_examples)
    .rename("examples")
)

genre_audit = (
    genre_audit
    .join(examples)
    .reset_index()
)

In [38]:
genre_audit = genre_audit.sort_values(
    "total_count",
    ascending=False
)

genre_audit.to_csv(
    "data/processed/genre_tag_audit.csv",
    index=False
)
# will stop here and continue with clean v2 notebook